# PointDiT in motion, Kaggle runner

Predicts PointDiT depth on the TUM RGB-D subset, either the baseline run or the sampling and sweep runs selected by the flags below. Runs on a Kaggle GPU (two T4) with Internet on.

## Setup

| Item | Kind | Notes |
|---|---|---|
| PointDiT TUM RGB-D Subset | dataset | always attached |
| Runner output with `predictions/baseline/run.json` | notebook output | needed for `RUN_SAMPLING` and `RUN_SWEEP` |
| `GITHUB_TOKEN` | Kaggle secret | fine-grained, read-only Contents, only while the repo is private |
| `HF_TOKEN` | Kaggle secret | access to `facebook/dinov3-vitl16-pretrain-lvd1689m`, plus `vitb16` for model B |
| `DINOV3_COMMIT` | setting | pinned DINOv3 commit |
| `OUTPUT_DIR` | setting | where smoke test, previews and checks write |
| `SWEEP_CONFIGS` | setting | `configs/sweep/<name>.yaml` files, run in order |

## Run flags

Both flags default to False, which runs the baseline prediction and previews.

| Flag | What it runs | Typical runtime |
|---|---|---|
| `RUN_SAMPLING` | `noise_seed0`, `noise_seed1`, `noise_seed2` and `zeros_rerun` on `configs/sampling_subsets.yaml`, then a determinism check of `zeros_rerun` against the baseline | not recorded |
| `RUN_SWEEP` | `run_sweep.py` on `SWEEP_CONFIGS` over `configs/sweep_subsets.yaml`, one config at a time on GPU 0, then a determinism check of `sweep_baseline_check` | not recorded |

The sampling check stops the notebook on a mismatch, while the sweep check only prints a warning. A failing sweep config is recorded and the next one runs.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
DINOV3_COMMIT = "6876159a11b4df116f30f667f8c9888617df0751"
OUTPUT_DIR = "/kaggle/working/outputs"
PROJECT_BRANCH = "main"
RUN_SAMPLING = False
RUN_SWEEP = False
# configs/sweep/<name>.yaml, run in this order.
SWEEP_CONFIGS = ["baseline_check", "steps_1", "steps_2", "steps_4", "fp16", "tokens_512", "tokens_2048", "model_b",
                 "downscale_2"]

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
DINOV3_DIR = f"{FORK_DIR}/third_party/dinov3"

## Clone the project

Clones the repo with its submodules and prints the commits.

In [ ]:
import base64
import os

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Passed to git through env vars, so the token never shows in commands, output or .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

# Check .git, since after a failed clone the DINOv3 clone below creates PROJECT_DIR.
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"

## DINOv3 source

Clones DINOv3 into the fork and checks out `DINOV3_COMMIT`.

In [ ]:
if not os.path.isdir(DINOV3_DIR):
    !git clone -q https://github.com/facebookresearch/dinov3.git {DINOV3_DIR}
if DINOV3_COMMIT:
    !git -C {DINOV3_DIR} checkout -q {DINOV3_COMMIT}
else:
    print("DINOV3_COMMIT is empty; paste the hash below into the config cell to pin it.")
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

## Requirements

In [ ]:
!pip install -q -r {FORK_DIR}/requirements.txt -r {PROJECT_DIR}/requirements.txt

## Model weights

Sets up the model L weights, and the model B weights only when a sweep config needs them.

In [ ]:
import subprocess
import sys

import yaml
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python {PROJECT_DIR}/scripts/setup_weights.py --models L

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.paths import checkpoint_path, dinov3_weights_path


def sweep_uses_model_b() -> bool:
    paths = [f"{PROJECT_DIR}/configs/sweep/{name}.yaml" for name in SWEEP_CONFIGS]
    return any((yaml.safe_load(open(p)).get("inference") or {}).get("model") == "B" for p in paths if os.path.isfile(p))


# A failed B setup only fails the model B config, which the sweep records.
if RUN_SWEEP and sweep_uses_model_b() and not (dinov3_weights_path("B").is_file() and checkpoint_path("B").is_file()):
    done = subprocess.run([sys.executable, f"{PROJECT_DIR}/scripts/setup_weights.py", "--models", "B"],
                          cwd=PROJECT_DIR, capture_output=True, text=True)
    print(done.stdout + done.stderr[-3000:])
    if done.returncode != 0:
        print(f"WARNING model B weights could not be set up (exit code {done.returncode}); "
              "sweep_model_b will fail and be recorded, the other configs are unaffected")
elif RUN_SWEEP:
    print("model B weights present or not needed by SWEEP_CONFIGS")

## Sequences

In [ ]:
import sys

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)

## Sanity checks

Runs the smoke test, prints versions, and checks sampler initialisation and Booster metrics against references.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/smoke_test.py --sequence "{sequences[0]}" --output-dir {OUTPUT_DIR}/smoke

In [ ]:
import torch

print("torch  ", torch.__version__)
print("CUDA   ", torch.version.cuda)
print("GPU    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

In [ ]:
REFERENCE_SEQUENCE = "rgbd_dataset_freiburg1_360"
REFERENCE_DIR = f"{PROJECT_DIR}/tests/fixtures/smoke_fr1360_664736a"

from pim.data.tum import read_rgb_list

ref_seq = next((s for s in sequences if s.name == REFERENCE_SEQUENCE), None)
if ref_seq is None:
    raise FileNotFoundError(f"{REFERENCE_SEQUENCE} not found under /kaggle/input")
first_frame = read_rgb_list(ref_seq)[0].path
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/check_sampler_init.py --image "{first_frame}" --reference "{REFERENCE_DIR}"

In [ ]:
EVAL_DATA_DIR = "/kaggle/working/datasets/eval"
BOOSTER_DIR = f"{EVAL_DATA_DIR}/Booster"

if not os.path.isdir(BOOSTER_DIR):
    from huggingface_hub import hf_hub_download

    hf_hub_download("haofeixu/pointdit-eval", "Booster.zip", repo_type="dataset", local_dir=EVAL_DATA_DIR)
    !cd {EVAL_DATA_DIR} && unzip -qo Booster.zip
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/check_booster_metrics.py --booster-dir {BOOSTER_DIR}

## Prediction helpers

Defines the `predict.py` command and runs command queues on both GPUs in parallel.

In [ ]:
import json
import os
import subprocess
import sys
import threading
from pathlib import Path

import yaml

from pim.data.tum import apply_window, load_sequence

SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"
PREDICTIONS_DIR = "/kaggle/working/predictions"

data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()


def predict_command(config: str, *extra: str, out: str | Path = PREDICTIONS_DIR) -> list[str]:
    return [sys.executable, f"{PROJECT_DIR}/scripts/predict.py", "--config", config,
            "--sequences-config", SEQUENCES_CONFIG, "--data-root", str(DATA_ROOT),
            "--out", str(out), "--device", "cuda:0", *extra]


def relay(proc: subprocess.Popen, prefix: str) -> int:
    for line in proc.stdout:
        print(f"{prefix}{line}", end="", flush=True)
    return proc.wait()


def run_on_gpus(queues: dict[int, list[list[str]]]) -> None:
    """Each GPU runs its commands one after another, the GPUs in parallel; a GPU stops at its first failure."""
    failed = {}

    def work(gpu: int, commands: list[list[str]]) -> None:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        for cmd in commands:
            proc = subprocess.Popen(cmd, env=env, cwd=PROJECT_DIR, text=True,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            code = relay(proc, f"[gpu{gpu}] ")
            if code != 0:
                failed[gpu] = code
                return

    threads = [threading.Thread(target=work, args=item, daemon=True) for item in queues.items()]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if failed:
        raise RuntimeError(f"predict.py failed on {failed} (gpu: exit code); see the output above")


def run_script(*cmd: object) -> None:
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if relay(proc, "") != 0:
        raise RuntimeError(f"{cmd[0]} exited with code {proc.returncode}; see the output above")


## Baseline prediction

Splits the four sequences over two GPUs and checks every frame was written. Skipped with `RUN_SAMPLING` or `RUN_SWEEP`.

In [ ]:
if RUN_SAMPLING or RUN_SWEEP:
    print("RUN_SAMPLING or RUN_SWEEP is True; skipping the baseline prediction")
else:
    RUN_CONFIG = f"{PROJECT_DIR}/configs/baseline.yaml"

    names = list(yaml.safe_load(open(SEQUENCES_CONFIG))["sequences"])
    frame_counts = {n: len(apply_window(load_sequence(DATA_ROOT / n, SEQUENCES_CONFIG))) for n in names}
    if len(names) != 4:
        raise RuntimeError(f"expected four sequences to split over two GPUs, got {names}")
    # Largest with smallest is the most even two-and-two split.
    by_size = sorted(names, key=frame_counts.get, reverse=True)
    assignment = {0: [by_size[0], by_size[3]], 1: [by_size[1], by_size[2]]}

    for gpu, seq_names in assignment.items():
        print(f"gpu{gpu}: {seq_names} ({sum(frame_counts[n] for n in seq_names)} frames)")
    run_on_gpus({gpu: [predict_command(RUN_CONFIG, "--sequences", *seq_names)] for gpu, seq_names in assignment.items()})

    run_dir = Path(PREDICTIONS_DIR) / yaml.safe_load(open(RUN_CONFIG))["name"]
    run = json.loads((run_dir / "run.json").read_text())
    print(f"\n{run_dir}: {run['start_time']} -> {run['end_time']}, "
          f"{run['frames_predicted']} predicted, {run['frames_skipped']} skipped")
    for name in names:
        seq = run["sequences"].get(name)
        files = len(list((run_dir / name).glob("*.npz")))
        if seq is None or files != seq["frames"] or seq["frames"] != frame_counts[name]:
            raise RuntimeError(f"{name}: run.json entry {seq}, {files} npz files, {frame_counts[name]} frames in window")
        print(f"  {name}: window {seq['window']}, {files} files, "
              f"mean latency {seq['mean_latency_ms']:.0f} ms, peak {seq['peak_mem_gb']:.2f} GB")
    for inv in run["invocations"]:
        commits = ", ".join(f"{k} {v['commit'][:7]}{'+dirty' if v['dirty'] else ''}" for k, v in inv["commits"].items())
        print(f"  run {inv['id']}: GPU {inv['cuda_visible_devices']} {inv['gpu']}, torch {inv['torch']}, "
              f"CUDA {inv['cuda']}, {inv['start_time']} -> {inv['end_time']}, {commits}")


## Baseline previews

In [ ]:
if RUN_SAMPLING or RUN_SWEEP:
    print("RUN_SAMPLING or RUN_SWEEP is True; skipping the previews")
else:
    PREVIEW_FRAMES = {
        "rgbd_dataset_freiburg1_desk": [209, 359, 508],
        "rgbd_dataset_freiburg1_room": [366, 516, 665],
        "rgbd_dataset_freiburg1_360": [0, 150, 299],
        "rgbd_dataset_freiburg3_long_office_household": [637, 787, 936],
    }
    for seq, frames in PREVIEW_FRAMES.items():
        frames_arg = " ".join(map(str, frames))
        !python {PROJECT_DIR}/scripts/preview_predictions.py --predictions /kaggle/working/predictions/baseline --data-root "{DATA_ROOT}" --sequences-config {PROJECT_DIR}/configs/sequences.yaml --sequence {seq} --frames {frames_arg} --out /kaggle/working/outputs/preview


## Sampling predictions

Runs with `RUN_SAMPLING`, two configs per T4 one after the other.

In [ ]:
SUBSETS = f"{PROJECT_DIR}/configs/sampling_subsets.yaml"
# Two configs per T4, one after the other.
SAMPLING_CONFIGS = {0: ["noise_seed0", "noise_seed1"], 1: ["noise_seed2", "zeros_rerun"]}

if RUN_SAMPLING:
    from pim.subsets import load_subsets

    subsets = load_subsets(Path(SUBSETS))
    for gpu, configs in SAMPLING_CONFIGS.items():
        print(f"gpu{gpu}: {configs}, {sum(r.last - r.first + 1 for r in subsets.values())} frames each")
    run_on_gpus({gpu: [predict_command(f"{PROJECT_DIR}/configs/{name}.yaml", "--subsets", SUBSETS) for name in configs]
                 for gpu, configs in SAMPLING_CONFIGS.items()})

    for name in (n for configs in SAMPLING_CONFIGS.values() for n in configs):
        run_dir = Path(PREDICTIONS_DIR) / name
        run = json.loads((run_dir / "run.json").read_text())
        print(f"\n{run_dir}: {run['config']['inference']}, {run['start_time']} -> {run['end_time']}")
        if run.get("subsets") != {seq: r.as_dict() for seq, r in subsets.items()}:
            raise RuntimeError(f"{name}: run.json subsets {run.get('subsets')} differ from {SUBSETS}")
        for seq, frame_range in subsets.items():
            entry = run["sequences"].get(seq)
            files = len(list((run_dir / seq).glob("*.npz")))
            expected = frame_range.last - frame_range.first + 1
            if entry is None or files != expected or entry["frames"] != expected:
                raise RuntimeError(f"{name} {seq}: run.json entry {entry}, {files} npz files, {expected} subset frames")
            # Latency from two parallel processes is not representative; see report/notes.md.
            print(f"  {seq}: frames {frame_range.first}-{frame_range.last}, {files} files, "
                  f"peak {entry['peak_mem_gb']:.2f} GB")
else:
    print("RUN_SAMPLING is False; skipping the sampling predictions")


## Determinism check

Runs with `RUN_SAMPLING` and stops if `zeros_rerun` differs from the attached baseline by more than 2e-3.

In [ ]:
def find_input_runs(name: str) -> list[Path]:
    """Run directories under /kaggle/input whose run.json has this run name, following symlinked folders."""
    found = []
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        dirs[:] = [d for d in dirs if d not in ("rgb", "depth")]
        if "run.json" in files and json.loads((Path(root) / "run.json").read_text()).get("name") == name:
            found.append(Path(root))
    return found


if RUN_SAMPLING:
    baselines = find_input_runs("baseline")
    if len(baselines) != 1:
        raise RuntimeError(f"expected one baseline run.json under /kaggle/input, found {[str(p) for p in baselines]}; "
                           "attach the output of the runner notebook version that wrote predictions/baseline")
    print(f"zeros_rerun vs {baselines[0]}")
    run_script(f"{PROJECT_DIR}/scripts/check_determinism.py", "--predictions", Path(PREDICTIONS_DIR) / "zeros_rerun",
               "--reference", baselines[0], "--subsets", SUBSETS, "--out", f"{OUTPUT_DIR}/sampling/determinism.csv")
else:
    print("RUN_SAMPLING is False; skipping the determinism check")


## Compute sweep

Runs with `RUN_SWEEP` and writes `sweep_latency.csv` and `determinism.csv` to `outputs/sweep/`. A determinism mismatch only prints a warning.

In [ ]:
SWEEP_SUBSETS = f"{PROJECT_DIR}/configs/sweep_subsets.yaml"
SWEEP_DIR = Path(PREDICTIONS_DIR) / "sweep"
SWEEP_OUT = Path(OUTPUT_DIR) / "sweep"
# run_sweep.py exit code when only the final determinism check fails.
DETERMINISM_MISMATCH = 3

if RUN_SWEEP:
    # Before any GPU work, so a missing attachment cannot waste the session.
    baselines = find_input_runs("baseline")
    if len(baselines) != 1:
        raise RuntimeError(f"expected one baseline run.json under /kaggle/input, found {[str(p) for p in baselines]}; "
                           "attach the output of the runner notebook version that wrote predictions/baseline")
    # A failing config is recorded and skipped.
    proc = subprocess.Popen(
        [sys.executable, f"{PROJECT_DIR}/scripts/run_sweep.py",
         "--configs", *(f"{PROJECT_DIR}/configs/sweep/{name}.yaml" for name in SWEEP_CONFIGS),
         "--subsets", SWEEP_SUBSETS, "--data-root", str(DATA_ROOT), "--sequences-config", SEQUENCES_CONFIG,
         "--out", str(SWEEP_DIR), "--reference", str(baselines[0]),
         "--summary-out", str(SWEEP_OUT / "sweep_latency.csv"), "--determinism-out", str(SWEEP_OUT / "determinism.csv")],
        cwd=PROJECT_DIR, text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    code = relay(proc, "")
    print("\nto download:")
    for path in sorted([*SWEEP_DIR.glob("*/sweep_*"), *SWEEP_DIR.glob("*/run.json"), *SWEEP_OUT.glob("*")]):
        print(f"  {path}")
    if code == DETERMINISM_MISMATCH:
        print("\nWARNING the determinism check failed (see the banner above); all sweep outputs are kept")
    elif code != 0:
        raise RuntimeError(f"run_sweep.py exited with code {code}; outputs written so far are kept, see above")
else:
    print("RUN_SWEEP is False; skipping the compute sweep")